# Restart File Output and Warm-Start Settings

This notebook validates the restart-file API split used by warm-start workflows:

1. Configure a copied plan to write a HEC-RAS restart file.
2. Run that plan through `RasCmdr.compute_plan()`.
3. Configure a copied unsteady-flow file and plan to use the generated restart file.
4. Run the warm-start continuation and compare the executed time window and runtime.

The example uses the official Muncie 2D unsteady project from `RasExamples`. Generated HEC-RAS project files and results are written outside the tracked examples folder.

In [1]:
# =============================================================================
# DEVELOPMENT MODE TOGGLE
# =============================================================================
from pathlib import Path
import os
import sys

USE_LOCAL_SOURCE = True

repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
if USE_LOCAL_SOURCE:
    local_path = str(repo_root)
    if local_path not in sys.path:
        sys.path.insert(0, local_path)
    print(f"LOCAL SOURCE MODE: Loading from {local_path}/ras_commander")
else:
    print("PIP PACKAGE MODE: Loading installed ras-commander")

from datetime import datetime
from time import perf_counter
import logging

import pandas as pd
from IPython.display import display

from ras_commander import RasCmdr, RasExamples, RasPlan, RasUnsteady, init_ras_project, ras
import ras_commander

# Keep notebook output focused on API results instead of full ras-commander logs.
logging.getLogger().setLevel(logging.WARNING)
for logger_name in list(logging.root.manager.loggerDict):
    if logger_name.startswith("ras_commander"):
        logging.getLogger(logger_name).setLevel(logging.WARNING)

print(f"Loaded ras-commander {ras_commander.__version__} from {ras_commander.__file__}")

LOCAL SOURCE MODE: Loading from C:\GH\symphony-workspaces\ras-commander\CLB-318/ras_commander


Loaded ras-commander 0.89.2 from C:\GH\symphony-workspaces\ras-commander\CLB-318\ras_commander\__init__.py


## Parameters

The default artifact root targets the CLB Symphony review folder. Set `RAS_COMMANDER_NOTEBOOK_WORKDIR` before execution to place generated projects somewhere else.

In [2]:
PROJECT_NAME = "Muncie"
RAS_VERSION = "7.0"
SOURCE_TEMPLATE_PLAN = "01"
RUN_SUFFIX = "restart_api"
NUM_CORES = 2

restart_datetime = ("02JAN1900", "1200")
warm_start_begin = datetime(1900, 1, 2, 12, 0)
warm_start_end = datetime(1900, 1, 3, 0, 0)

artifact_root = Path(
    os.environ.get(
        "RAS_COMMANDER_NOTEBOOK_WORKDIR",
        r"H:/Symphony/ras-commander/CLB-318/notebook_runs",
    )
)
if not artifact_root.drive or not Path(artifact_root.drive + "/").exists():
    artifact_root = repo_root / "working" / "notebook_runs"
artifact_root.mkdir(parents=True, exist_ok=True)

print(f"Artifact root: {artifact_root}")

Artifact root: H:\Symphony\ras-commander\CLB-318\notebook_runs


## Extract and Initialize the Example Project

`RasExamples.extract_project()` gives the notebook a fresh project folder on each run, keeping the committed example assets immutable.

In [3]:
project_path = RasExamples.extract_project(
    PROJECT_NAME,
    output_path=artifact_root,
    suffix=RUN_SUFFIX,
)
init_ras_project(project_path, RAS_VERSION)

print(f"Project folder: {project_path}")
print(f"HEC-RAS executable: {ras.ras_exe_path}")

display_columns = [
    "plan_number",
    "unsteady_number",
    "Simulation Date",
    "Write IC File",
    "IC Time",
]
display(ras.plan_df[[c for c in display_columns if c in ras.plan_df.columns]])

unsteady_columns = ["unsteady_number", "Use Restart", "Restart Filename"]
display(ras.unsteady_df[[c for c in unsteady_columns if c in ras.unsteady_df.columns]])

Project folder: H:\Symphony\ras-commander\CLB-318\notebook_runs\Muncie_restart_api
HEC-RAS executable: C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe


,plan_number,unsteady_number,Simulation Date,Write IC File,IC Time
0,01,01,"02JAN1900,0000,02JAN1900,2400",0,",,"
1,03,01,"02JAN1900,0000,02JAN1900,2400",0,",,"
2,04,01,"02JAN1900,0000,02JAN1900,2400",0,",,"


,unsteady_number,Use Restart,Restart Filename
0,01,0,Muncie.p05.02JAN1900 1200.rst


## Configure Restart Output on a Copied Plan

`RasPlan.set_restart_output_settings()` writes the plan-file `Write IC File` block. This controls creation of restart files only; it does not turn on `Use Restart` in the unsteady-flow file.

In [4]:
source_plan = RasPlan.clone_plan(
    SOURCE_TEMPLATE_PLAN,
    new_shortid="RST_Source",
    new_title="Restart output source",
)

RasPlan.set_restart_output_settings(
    source_plan,
    save_datetime=restart_datetime,
    write_at_sim_end=False,
)

source_output_settings = RasPlan.get_restart_output_settings(source_plan)
restart_file = project_path / source_output_settings["expected_filename"]

print(f"Restart-output plan: p{source_plan}")
print(f"Expected restart file: {restart_file.name}")
display(pd.DataFrame([source_output_settings]).drop(columns=["raw", "compatibility_note"]))
display(pd.DataFrame([source_output_settings["raw"]]))

assert source_output_settings["enabled"] is True
assert source_output_settings["save_at_fixed_datetime"] is True
assert source_output_settings["save_datetime"] == "02JAN1900,1200"

Restart-output plan: p02
Expected restart file: Muncie.p02.02JAN1900 1200.rst


,enabled,save_at_fixed_datetime,save_time_hours,save_date,save_time,save_datetime,recurrence_interval_hours,write_at_sim_end,expected_filename,output_filename_pattern
0,True,True,None,02JAN1900,1200,"02JAN1900,1200",None,False,Muncie.p02.02JAN1900 1200.rst,ProjectName.p##.DDMMMYYYY hhmm.rst


,Write IC File,Write IC File at Fixed DateTime,IC Time,Write IC File Reoccurance,Write IC File at Sim End
0,1,-1,",02JAN1900,1200",,0


## Run the Restart-Output Source Plan

The plan is executed through `RasCmdr.compute_plan()`, which is the ras-commander API layer for HEC-RAS command-line execution.

In [5]:
start = perf_counter()
source_result = RasCmdr.compute_plan(
    source_plan,
    num_cores=NUM_CORES,
    verify=True,
    force_rerun=True,
)
source_seconds = perf_counter() - start

assert bool(source_result), "Restart-output source plan did not verify successfully"
assert restart_file.exists(), f"Expected restart file was not created: {restart_file}"

restart_file_info = pd.DataFrame([
    {
        "plan": source_plan,
        "restart_file": restart_file.name,
        "size_bytes": restart_file.stat().st_size,
        "runtime_seconds": round(source_seconds, 2),
    }
])
display(restart_file_info)

,plan,restart_file,size_bytes,runtime_seconds
0,02,Muncie.p02.02JAN1900 1200.rst,29396,13.86


## Configure a Warm-Start Plan to Use the Restart File

`RasUnsteady.set_restart_settings()` writes `Use Restart` and `Restart Filename` in a copied unsteady-flow file. The continuation plan is then pointed at that unsteady-flow file and starts at the restart timestamp.

In [6]:
warm_unsteady = RasPlan.clone_unsteady(
    "01",
    new_title="Warm restart usage",
)
RasUnsteady.set_restart_settings(
    warm_unsteady,
    use_restart=True,
    restart_filename=restart_file.name,
)
warm_usage_settings = RasUnsteady.get_restart_settings(warm_unsteady)

warm_plan = RasPlan.clone_plan(
    source_plan,
    new_shortid="Warm_Start",
    new_title="Warm-start continuation",
    unsteady_flow=warm_unsteady,
)
RasPlan.update_simulation_date(
    warm_plan,
    warm_start_begin,
    warm_start_end,
)
RasPlan.set_restart_output_settings(warm_plan, enabled=False)

warm_output_settings = RasPlan.get_restart_output_settings(warm_plan)
warm_plan_date = RasPlan.get_plan_value(warm_plan, "Simulation Date")

print(f"Warm-start unsteady file: u{warm_unsteady}")
print(f"Warm-start plan: p{warm_plan}")
print(f"Warm-start simulation date: {warm_plan_date}")
display(pd.DataFrame([warm_usage_settings]).drop(columns=["raw", "compatibility_note"]))
display(pd.DataFrame([warm_usage_settings["raw"]]))
display(pd.DataFrame([warm_output_settings]).drop(columns=["raw", "compatibility_note"]))

assert warm_usage_settings["use_restart"] is True
assert warm_usage_settings["restart_filename"] == restart_file.name
assert warm_output_settings["enabled"] is False
assert warm_plan_date == "02JAN1900,1200,03JAN1900,0000"

Warm-start unsteady file: u02
Warm-start plan: p05
Warm-start simulation date: 02JAN1900,1200,03JAN1900,0000


,use_restart,restart_filename
0,True,Muncie.p02.02JAN1900 1200.rst


,Use Restart,Restart Filename
0,-1,Muncie.p02.02JAN1900 1200.rst


,enabled,save_at_fixed_datetime,save_time_hours,save_date,save_time,save_datetime,recurrence_interval_hours,write_at_sim_end,expected_filename,output_filename_pattern
0,False,False,None,None,None,None,None,False,None,ProjectName.p##.DDMMMYYYY hhmm.rst


## Run the Warm-Start Continuation

The warm-start run uses the restart file and solves only the second half of the event. The summary compares the full source run against the continuation run and records the skipped spin-up period.

In [7]:
start = perf_counter()
warm_result = RasCmdr.compute_plan(
    warm_plan,
    num_cores=NUM_CORES,
    verify=True,
    force_rerun=True,
)
warm_seconds = perf_counter() - start

assert bool(warm_result), "Warm-start continuation plan did not verify successfully"

summary = pd.DataFrame([
    {
        "workflow": "restart-output source",
        "plan": source_plan,
        "uses_restart_file": False,
        "hours_simulated": 24,
        "runtime_seconds": round(source_seconds, 2),
        "success": bool(source_result),
    },
    {
        "workflow": "warm-start continuation",
        "plan": warm_plan,
        "uses_restart_file": True,
        "hours_simulated": 12,
        "runtime_seconds": round(warm_seconds, 2),
        "success": bool(warm_result),
    },
])
summary["seconds_per_simulated_hour"] = (
    summary["runtime_seconds"] / summary["hours_simulated"]
).round(2)

spinup_hours_skipped = 12
runtime_delta_seconds = round(source_seconds - warm_seconds, 2)
print(f"Spin-up period skipped by restart workflow: {spinup_hours_skipped} hours")
print(f"Runtime difference observed on this workstation: {runtime_delta_seconds} seconds")
display(summary)

Spin-up period skipped by restart workflow: 12 hours
Runtime difference observed on this workstation: 2.66 seconds


,workflow,plan,uses_restart_file,hours_simulated,runtime_seconds,success,seconds_per_simulated_hour
0,restart-output source,02,False,24,13.86,True,0.58
1,warm-start continuation,05,True,12,11.21,True,0.93


## Round-Trip Parse Check

The final parse confirms that restart output creation remains in the plan file and restart usage remains in the unsteady-flow file.

In [8]:
ras.plan_df = ras.get_plan_entries()
ras.unsteady_df = ras.get_unsteady_entries()

round_trip = pd.DataFrame([
    {
        "file": f"p{source_plan}",
        "api": "RasPlan.get_restart_output_settings",
        "enabled": source_output_settings["enabled"],
        "save_datetime": source_output_settings["save_datetime"],
        "restart_filename": source_output_settings["expected_filename"],
    },
    {
        "file": f"u{warm_unsteady}",
        "api": "RasUnsteady.get_restart_settings",
        "enabled": warm_usage_settings["use_restart"],
        "save_datetime": None,
        "restart_filename": warm_usage_settings["restart_filename"],
    },
])

display(round_trip)

display(ras.plan_df[ras.plan_df["plan_number"].isin([source_plan, warm_plan])][[
    "plan_number",
    "unsteady_number",
    "Simulation Date",
    "Write IC File",
    "Write IC File at Fixed DateTime",
    "IC Time",
    "Write IC File Reoccurance",
    "Write IC File at Sim End",
]])

display(ras.unsteady_df[ras.unsteady_df["unsteady_number"].isin([warm_unsteady])][[
    "unsteady_number",
    "Use Restart",
    "Restart Filename",
]])

,file,api,enabled,save_datetime,restart_filename
0,p02,RasPlan.get_restart_output_settings,True,"02JAN1900,1200",Muncie.p02.02JAN1900 1200.rst
1,u02,RasUnsteady.get_restart_settings,True,None,Muncie.p02.02JAN1900 1200.rst


,plan_number,unsteady_number,Simulation Date,Write IC File,Write IC File at Fixed DateTime,IC Time,Write IC File Reoccurance,Write IC File at Sim End
3,02,01,"02JAN1900,0000,02JAN1900,2400",1,-1,",02JAN1900,1200",,0
4,05,02,"02JAN1900,1200,03JAN1900,0000",0,0,",,",,0


,unsteady_number,Use Restart,Restart Filename
1,02,-1,Muncie.p02.02JAN1900 1200.rst
